# Feature Engineering

## Objective

This notebook demonstrates the first version of the traditional machine learning feature engineering pipeline.

The engineered features include:

- Rolling Mean
- Rolling Standard Deviation
- Lag Features
- Rate of Change

All features are generated independently for each engine to preserve the temporal structure of the NASA C-MAPSS dataset.

This notebook evaluates:

- Number of generated features
- Example engineered features
- NaN values introduced
- Engineering decisions for the next pipeline stage

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

A:\AI Engineer in the way\ML Projects\Predictive-Maintenance-RUL


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from src.data.loader import DataLoader
from src.data.validator import DataValidator

from src.config.config import (
    TRAIN_DATA_PATH,
    TEST_DATA_PATH,
    RUL_DATA_PATH
)

In [3]:
loader = DataLoader(
    train_path=TRAIN_DATA_PATH,
    test_path=TEST_DATA_PATH,
    rul_path=RUL_DATA_PATH,
)

train_df = loader.load_train()
test_df = loader.load_test()
rul_df = loader.load_rul()

2026-07-23 19:21:37 | INFO | loader.py | Line:18 | Reading train_FD004.txt
2026-07-23 19:21:39 | INFO | loader.py | Line:21 | train_FD004.txt Loaded Successfully
2026-07-23 19:21:39 | INFO | loader.py | Line:18 | Reading test_FD004.txt
2026-07-23 19:21:40 | INFO | loader.py | Line:21 | test_FD004.txt Loaded Successfully
2026-07-23 19:21:40 | INFO | loader.py | Line:18 | Reading RUL_FD004.txt
2026-07-23 19:21:40 | INFO | loader.py | Line:21 | RUL_FD004.txt Loaded Successfully


In [4]:
import pandas as pd

from src.preprocessing.rul_generator import RULGenerator
from src.preprocessing.feature_engineer import FeatureEngineer
from src.utils.constant import SENSOR_COLUMNS

In [5]:
generator = RULGenerator(train_df)

train_df = generator.generate(cap=125)

train_df.head()

2026-07-23 19:21:52 | INFO | rul_generator.py | Line:82 | Generating Remaining Useful Life (RUL)...
2026-07-23 19:21:52 | INFO | rul_generator.py | Line:92 | Applying RUL cap = 125
2026-07-23 19:21:52 | INFO | rul_generator.py | Line:96 | RUL generated successfully.


,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670,125
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552,125
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213,125
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176,125
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754,125


In [6]:
engineer = FeatureEngineer(
    sensor_columns=SENSOR_COLUMNS,
    rolling_window=5,
    lags=[1,2,3]
)

feature_df = engineer.transform(train_df)

2026-07-23 19:22:11 | INFO | feature_engineer.py | Line:50 | Starting Feature Engineering...
2026-07-23 19:22:11 | INFO | feature_engineer.py | Line:119 | Generating Rolling Mean features...
2026-07-23 19:22:12 | INFO | feature_engineer.py | Line:148 | Generating Rolling Std features...
2026-07-23 19:22:14 | INFO | feature_engineer.py | Line:179 | Generating Lag Features...
A:\AI Engineer in the way\ML Projects\Predictive-Maintenance-RUL\src\preprocessing\feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
2026-07-23 19:22:14 | INFO | feature_engineer.py | Line:205 | Generating Rate of Change features...
A:\AI Engineer in the way\ML Projects\Predictive-Maintenance-RUL\src\preprocessing\feature_engineer.py:213: Perform

In [7]:
print("Before :", train_df.shape)

print("After  :", feature_df.shape)

Before : (61249, 27)
After  : (61249, 153)


In [8]:
new_features = feature_df.shape[1] - train_df.shape[1]

print(f"New Features: {new_features}")

New Features: 126


In [9]:
engine_1 = feature_df[
    feature_df["unit_number"] == 1
]

columns = [
    "time_in_cycles",
    "sensor_2",
    "sensor_2_roll_mean_5",
    "sensor_2_roll_std_5",
    "sensor_2_lag_1",
    "sensor_2_diff",
]

engine_1[columns].head(15)

,time_in_cycles,sensor_2,sensor_2_roll_mean_5,sensor_2_roll_std_5,sensor_2_lag_1,sensor_2_diff
0,1,549.68,549.680000,NaN,NaN,NaN
1,2,606.07,577.875000,39.873751,549.68,56.39
2,3,548.95,568.233333,32.769547,606.07,-57.12
3,4,548.70,563.350000,28.483035,548.95,-0.25
4,5,536.10,557.900000,27.513178,548.70,-12.60
5,6,554.77,558.918000,27.225662,536.10,18.67
6,7,641.83,566.070000,42.896037,554.77,87.06
7,8,549.05,566.090000,42.886081,641.83,-92.78
8,9,549.55,566.260000,42.801515,549.05,0.50
9,10,536.35,566.310000,42.757598,549.55,-13.20


In [11]:
nan_counts = (
    feature_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

nan_counts[nan_counts > 0].head(20)

sensor_13_lag_3    747
sensor_15_lag_3    747
sensor_16_lag_3    747
sensor_1_lag_3     747
sensor_17_lag_3    747
sensor_21_lag_3    747
sensor_20_lag_3    747
sensor_19_lag_3    747
sensor_18_lag_3    747
sensor_7_lag_3     747
sensor_8_lag_3     747
sensor_9_lag_3     747
sensor_10_lag_3    747
sensor_11_lag_3    747
sensor_12_lag_3    747
sensor_14_lag_3    747
sensor_6_lag_3     747
sensor_5_lag_3     747
sensor_2_lag_3     747
sensor_3_lag_3     747
dtype: int64

In [12]:
total_nan = feature_df.isna().sum().sum()

print(total_nan)

41832


# Engineering Decisions

## Observations

- 126 engineered features were successfully created.
- Original observations were preserved.
- The number of rows remained unchanged.
- Feature engineering was performed independently for each engine.
- NaN values appear only where expected.

## Decisions

- NaN values will be handled in the preprocessing pipeline (Sprint 7).
- No feature selection will be performed at this stage.
- No scaling is applied in this sprint.
- The FeatureEngineer remains deterministic and reusable.